# EDA

Цель — понять, какие признаки связаны с отменой бронирования, найти сильные зависимости и сформировать гипотезы для модели.

Важно: здесь смотрим **связи**, а не доказываем причины.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet("../data/interim/hotel_bookings_clean.parquet")

print(df.shape)
df.head()

## 1. Target

In [ ]:
target_stats = pd.DataFrame({
    "count": df["is_canceled"].value_counts(),
    "share": df["is_canceled"].value_counts(normalize=True)
})

target_stats

In [ ]:
sns.countplot(data=df, x="is_canceled")
plt.title("Распределение target")
plt.xlabel("Is canceled")
plt.ylabel("Количество бронирований")
plt.show()

### Вывод

Около **37% броней отменяются**, 63% — нет.

Дисбаланс есть, но он не сильный. Пока ничего балансировать не буду.  
Временной split нужен отдельно — потому что данные зависят от времени, а не из-за дисбаланса классов.

## 2. Категориальные признаки vs target

Для категорий важно смотреть не только долю отмен, но и **размер группы**.  
Если в категории 2–5 объектов и cancellation rate = 100%, нормальный вывод из этого делать нельзя.

In [ ]:
def cancellation_stats(col):
    return (
        df.groupby(col, observed=True)["is_canceled"]
        .agg(
            bookings="size",
            cancellations="sum",
            cancellation_rate="mean"
        )
        .sort_values("cancellation_rate", ascending=False)
    )

In [ ]:
categorical_cols = [
    "hotel",
    "meal",
    "market_segment",
    "distribution_channel",
    "deposit_type",
    "customer_type",
    "reserved_room_type",
    "is_repeated_guest",
    "has_children",
    "has_agent",
    "has_company",
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    display(cancellation_stats(col))

### Вывод

Есть несколько интересных признаков:

- `hotel`: в City Hotel отмен заметно больше, чем в Resort Hotel;
- `deposit_type`: `Non Refund` очень сильно связан с отменой;
- `has_agent`: при наличии агента доля отмен **выше**, а не ниже;
- `has_company`: при наличии компании доля отмен заметно ниже;
- у редких категорий типа `Undefined` или редких room type высокий rate может быть случайным — всегда смотрю на `bookings`.

Пока это только зависимости. Например, нельзя сказать, что агент **вызывает** отмену.

## 3. Самые интересные категориальные признаки

In [ ]:
important_cat = [
    "deposit_type",
    "market_segment",
    "customer_type",
    "hotel"
]

for col in important_cat:
    print(f"\n--- {col} ---")
    display(cancellation_stats(col))

### Вывод

Самый сильный результат здесь — `deposit_type`.

У `Non Refund` cancellation rate почти 100%, причём группа большая, поэтому это уже не эффект нескольких строк.

Это выглядит очень сильным сигналом для модели, но причина пока неизвестна. Нужно проверить, не связан ли `Non Refund` с определённым `market_segment`.

In [ ]:
deposit_by_segment = (
    df.groupby(["market_segment", "deposit_type"], observed=True)["is_canceled"]
    .agg(bookings="size", cancellation_rate="mean")
    .sort_values("bookings", ascending=False)
)

deposit_by_segment

## 4. Lead time

In [ ]:
df.groupby("is_canceled")["lead_time"].describe()

In [ ]:
sns.histplot(
    data=df,
    x="lead_time",
    hue="is_canceled",
    bins=50,
    element="step"
)

plt.title("Lead time vs target")
plt.show()

In [ ]:
lead_time_labels = [
    "0-7",
    "8-30",
    "31-90",
    "91-180",
    "181-365",
    "365+"
]

df["lead_time_group"] = pd.cut(
    df["lead_time"],
    bins=[-1, 7, 30, 90, 180, 365, np.inf],
    labels=lead_time_labels
)

lead_time_stats = (
    df.groupby("lead_time_group", observed=True)["is_canceled"]
    .agg(bookings="size", cancellation_rate="mean")
)

lead_time_stats

In [ ]:
lead_time_stats["cancellation_rate"].plot(kind="bar", figsize=(8, 4))

plt.ylabel("Cancellation rate")
plt.xlabel("Lead time")
plt.title("Cancellation rate by lead time")
plt.xticks(rotation=0)
plt.show()

### Вывод

Очень сильная и почти монотонная зависимость:

**чем раньше относительно даты заезда человек бронирует, тем выше вероятность отмены.**

Примерно:
- 0–7 дней → около 10% отмен;
- 91–180 → около 45%;
- 365+ → около 68%.

`lead_time` выглядит одним из самых сильных признаков для будущей модели.

## 5. ADR

In [ ]:
adr_99 = df["adr"].quantile(0.99)

sns.histplot(
    data=df[(df["adr"] >= 0) & (df["adr"] <= adr_99)],
    x="adr",
    hue="is_canceled",
    bins=50,
    element="step"
)

plt.title("ADR vs target, до 99 перцентиля")
plt.show()

In [ ]:
df.groupby("is_canceled")["adr"].agg(
    ["count", "mean", "median"]
)

Распределение `adr` не нормальное: оно скошено вправо, есть длинный хвост и сильные выбросы.

По одной медиане сильной зависимости от target не видно, поэтому дополнительно разобью ADR на интервалы.

In [ ]:
adr_for_analysis = df[
    (df["adr"] >= 0) &
    (df["adr"] <= adr_99)
].copy()

adr_for_analysis["adr_bin"] = pd.qcut(
    adr_for_analysis["adr"],
    q=10,
    duplicates="drop"
)

adr_stats = (
    adr_for_analysis
    .groupby("adr_bin", observed=True)["is_canceled"]
    .agg(bookings="size", cancellation_rate="mean")
)

adr_stats

In [ ]:
adr_stats["cancellation_rate"].plot(kind="bar", figsize=(10, 4))

plt.ylabel("Cancellation rate")
plt.xlabel("ADR group")
plt.title("Cancellation rate by ADR quantile")
plt.xticks(rotation=45, ha="right")
plt.show()

### Вывод

У `adr` есть выбросы, поэтому обычные средние и графики легко искажаются.

По медианам сильной разницы между отменёнными и неотменёнными бронями нет.  
Окончательный вывод делаю по группам ADR выше — если rate меняется с ценой, признак всё равно может быть полезен модели.

`adr = 0` пока не интерпретирую как персонал/служебные брони — данных для такого вывода нет.

## 6. Время

In [ ]:
month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

monthly_cancel = (
    df.groupby("arrival_date_month")["is_canceled"]
    .mean()
    .reindex(month_order)
)

monthly_cancel.plot(kind="bar", figsize=(10, 4))
plt.ylabel("Cancellation rate")
plt.title("Cancellation rate by month")
plt.show()

По месяцам разница есть, но по этому графику нельзя сразу говорить, есть сезонность или нет.

Причина: 2015 начинается летом, а 2017 заканчивается летом. Поэтому месяцы смешивают эффект месяца и эффект года.

In [ ]:
df["arrival_year_month"] = pd.to_datetime(
    df["arrival_date_year"].astype(str)
    + "-"
    + df["arrival_date_month"],
    format="%Y-%B"
)

monthly_timeline = (
    df.groupby("arrival_year_month")["is_canceled"]
    .mean()
    .sort_index()
)

monthly_timeline.plot(figsize=(12, 4))
plt.ylabel("Cancellation rate")
plt.xlabel("Arrival month")
plt.title("Cancellation rate over time")
plt.show()

### Вывод

Теперь можно смотреть на изменение cancellation rate во времени без смешивания годов.

Этот график также подтверждает, почему финальный train/validation/test нужно делить **по времени**, а не случайно.

## 7. Количество гостей и длительность проживания

In [ ]:
df["total_guests"] = (
    df["adults"]
    + df["children"].fillna(0)
    + df["babies"]
)

df["total_nights"] = (
    df["stays_in_week_nights"]
    + df["stays_in_weekend_nights"]
)

In [ ]:
guest_stats = (
    df.groupby("total_guests")["is_canceled"]
    .agg(bookings="size", cancellation_rate="mean")
    .query("bookings >= 100")
)

night_stats = (
    df.groupby("total_nights")["is_canceled"]
    .agg(bookings="size", cancellation_rate="mean")
    .query("bookings >= 100")
)

display(guest_stats)
display(night_stats)

### Вывод

Сравнивать только медианы было недостаточно — одинаковая медиана ещё не означает одинаковое распределение.

Поэтому здесь смотрю cancellation rate для каждого размера группы и количества ночей, но оставляю только группы хотя бы со 100 наблюдениями.

Если явной монотонной зависимости нет — это нормально: признаки всё равно могут быть полезны в комбинации с другими.

## 8. История клиента

In [ ]:
history_cols = [
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "is_repeated_guest"
]

for col in history_cols:
    print(f"\n--- {col} ---")
    display(
        df.groupby(col)["is_canceled"]
        .agg(bookings="size", cancellation_rate="mean")
    )

### Вывод

История клиента выглядит очень информативной.

- если раньше уже была хотя бы одна отмена, вероятность новой отмены сильно растёт;
- клиенты с успешными предыдущими бронированиями отменяют заметно реже;
- repeated guests отменяют гораздо реже новых клиентов.

Для больших значений `previous_cancellations` выборка маленькая, поэтому проценты 100% там нельзя воспринимать всерьёз.

Позже можно попробовать создать `has_previous_cancellation`.

## 9. Country

In [ ]:
top_countries = (
    df["country"]
    .value_counts()
    .head(15)
    .index
)

country_stats = (
    df[df["country"].isin(top_countries)]
    .groupby("country")["is_canceled"]
    .agg(bookings="size", cancellation_rate="mean")
    .sort_values("cancellation_rate", ascending=False)
)

country_stats

In [ ]:
df["country_group"] = np.where(
    df["country"] == "Unknown",
    "Unknown",
    np.where(
        df["country"] == "PRT",
        "Domestic",
        "Foreign"
    )
)

country_group_stats = (
    df.groupby("country_group")["is_canceled"]
    .agg(
        bookings="size",
        cancellations="sum",
        cancellation_rate="mean"
    )
    .sort_values("cancellation_rate", ascending=False)
)

country_group_stats

In [ ]:
country_group_stats["cancellation_rate"].plot(
    kind="bar",
    figsize=(7, 4)
)

plt.ylabel("Cancellation rate")
plt.xlabel("Country group")
plt.title("Cancellation rate by country group")
plt.xticks(rotation=0)
plt.show()

### Вывод

У бронирований из `PRT` cancellation rate заметно выше, чем у иностранных клиентов.

Это сильная зависимость, но пока нельзя говорить, **почему** она возникает.  
Возможно, дело частично в типе отеля, market segment или канале бронирования.

In [ ]:
country_hotel_stats = (
    df.groupby(["country_group", "hotel"], observed=True)["is_canceled"]
    .agg(bookings="size", cancellation_rate="mean")
)

country_hotel_stats

## Итог EDA

Самые интересные признаки на этом этапе:

- `lead_time` — очень сильная почти монотонная связь;
- `deposit_type` — `Non Refund` очень сильно связан с отменой;
- `previous_cancellations` — прошлые отмены дают сильный сигнал;
- `previous_bookings_not_canceled` — успешная история снижает риск;
- `is_repeated_guest` — повторные гости отменяют реже;
- `country_group` — `PRT` сильно отличается от иностранных клиентов;
- `hotel` — у City Hotel cancellation rate выше.

Что важно:

- редкие категории не оцениваю только по большому cancellation rate;
- не делаю причинные выводы из корреляций;
- выбросы пока не удаляю автоматически;
- временную структуру обязательно учту в split.

Следующий этап — **Feature Engineering + временной Train / Validation / Holdout split**.